# Threshold-Aligned Reliability Analysis

This notebook recomputes reliability quantities so that confidence/certainty is
consistent with each primary classifier's validation-selected threshold `tau`.

No primary RPI model is retrained. The required input is the compact artifact
generated by Notebook 01.

Two quantities are used:

- **Decision-class probability:** `p` when the operational prediction is positive,
  otherwise `1-p`; used for HC90 analysis.
- **Threshold-aligned decision certainty:** normalized distance from the
  validation-selected decision threshold; used for failure detection and selective
  prediction.




In [ ]:
from pathlib import Path
import json, math, platform, shutil, sys, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, matthews_corrcoef
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold

warnings.filterwarnings('ignore', category=FutureWarning)

INPUT_ROOT = Path('/kaggle/input/datasets/abdullahnayemwasi/dependable-rpi-journal-artifacts')
INPUT_FALLBACK_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working/rpi_reliability_threshold_aligned')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FIG_ROOT = OUTPUT_ROOT / 'figures'
FIG_SOURCE_ROOT = FIG_ROOT / 'source_data'
FIG_ROOT.mkdir(parents=True, exist_ok=True)
FIG_SOURCE_ROOT.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 17
N_META_SPLITS = 5
RF_PARAMS = dict(n_estimators=350, max_depth=6, min_samples_leaf=12, class_weight='balanced', n_jobs=-1)
COVERAGES = (1.00, 0.95, 0.90, 0.80, 0.70, 0.60, 0.50)
RUN = {'high_confidence':True,'failure_ablation':True,'secondary_leakage':True,'lodo':True,'risk_abstention':True,'make_figures':True}
print('Output root:', OUTPUT_ROOT)

In [ ]:
USECOLS=['row_id','r_idx','p_idx','y','dataset','protocol','seed','fold','model','run_id','pair_id','threshold','raw_probability','raw_prediction','wrong','raw_confidence','log1p_train_rna_degree','rna_centroid_cosine','protein_centroid_cosine','rna_diagonal_ood','protein_diagonal_ood']

def locate_artifact_source():
    candidates = []

    if INPUT_ROOT.exists():
        candidates.extend(INPUT_ROOT.rglob("dependable_rpi_journal_v1_ARTIFACTS.zip"))
        candidates.extend(INPUT_ROOT.rglob("*.zip"))

    # Kaggle mount naming sometimes differs from the URL-like path.
    if not candidates and INPUT_FALLBACK_ROOT.exists():
        candidates.extend(
            INPUT_FALLBACK_ROOT.rglob("dependable_rpi_journal_v1_ARTIFACTS.zip")
        )

    if candidates:
        # Prefer the exact expected filename.
        candidates = sorted(
            set(candidates),
            key=lambda p: (p.name != "dependable_rpi_journal_v1_ARTIFACTS.zip", len(str(p)))
        )
        return "zip", candidates[0]

    # Fallback: the dataset may contain extracted files.
    direct_candidates = []
    for root in [INPUT_ROOT, INPUT_FALLBACK_ROOT]:
        if root.exists():
            direct_candidates.extend(root.rglob("all_predictions_enriched.csv"))

    if direct_candidates:
        p = sorted(direct_candidates, key=lambda q: len(str(q)))[0]
        # p = .../analysis/all_predictions_enriched.csv.gz
        return "directory", p.parent.parent

    raise FileNotFoundError(
        "Could not find dependable_rpi_journal_v1_ARTIFACTS.zip or "
        "analysis/all_predictions_enriched.csv.gz under the Kaggle input mounts."
    )

SOURCE_KIND, ARTIFACT_SOURCE = locate_artifact_source()
print("Artifact source type:", SOURCE_KIND)
print("Artifact source:", ARTIFACT_SOURCE)

def read_artifact_csv(member, **kwargs):
    if SOURCE_KIND=='zip':
        with zipfile.ZipFile(ARTIFACT_SOURCE) as z:
            with z.open(member) as f:
                return pd.read_csv(f,compression='gzip' if member.endswith('.gz') else None,**kwargs)
    path=ARTIFACT_SOURCE/member
    return pd.read_csv(path,compression='gzip' if str(path).endswith('.gz') else None,**kwargs)

pred=read_artifact_csv('analysis/all_predictions_enriched.csv',usecols=USECOLS)
for c in ['seed','fold','y','raw_prediction','wrong']: pred[c]=pd.to_numeric(pred[c],downcast='integer')
for c in ['threshold','raw_probability','raw_confidence','log1p_train_rna_degree','rna_centroid_cosine','protein_centroid_cosine','rna_diagonal_ood','protein_diagonal_ood']:
    pred[c]=pd.to_numeric(pred[c],downcast='float')
print('Rows:',len(pred),'runs:',pred.run_id.nunique())
assert len(pred)==449244
assert pred.run_id.nunique()==180

In [ ]:
def locate_source():
    exact=[]
    if INPUT_ROOT.exists(): exact.extend(INPUT_ROOT.rglob('dependable_rpi_journal_v1_ARTIFACTS.zip'))
    if not exact and INPUT_FALLBACK_ROOT.exists(): exact.extend(INPUT_FALLBACK_ROOT.rglob('dependable_rpi_journal_v1_ARTIFACTS.zip'))
    if exact: return 'zip', sorted(exact,key=lambda p:len(str(p)))[0]
    direct=[]
    for root in [INPUT_ROOT,INPUT_FALLBACK_ROOT]:
        if root.exists(): direct.extend(root.rglob('all_predictions_enriched.csv'))
    if direct:
        q=sorted(direct,key=lambda p:len(str(p)))[0]
        return 'directory', q.parent.parent
    raise FileNotFoundError('Could not find primary benchmark artifact ZIP or enriched prediction file.')

SOURCE_KIND, ARTIFACT_SOURCE = locate_source()
print(SOURCE_KIND, ARTIFACT_SOURCE)

In [ ]:
EPS=1e-12
p=pred.raw_probability.to_numpy(float); yh=pred.raw_prediction.to_numpy(int); tau=pred.threshold.to_numpy(float)
calc=(p>=tau).astype(int)
assert np.array_equal(calc,yh)
argmax=(p>=0.5).astype(int)
pred['argmax_vs_operational_mismatch']=(argmax!=yh).astype(int)
pred['decision_class_probability']=np.where(yh==1,p,1-p)
pred['threshold_distance']=np.abs(p-tau)
pos=(p-tau)/np.maximum(1-tau,EPS)
neg=(tau-p)/np.maximum(tau,EPS)
pred['decision_certainty']=np.clip(np.where(yh==1,pos,neg),0,1)

audit=(pred.groupby(['dataset','model']).agg(n=('y','size'),mismatch_rate=('argmax_vs_operational_mismatch','mean'),mean_threshold=('threshold','mean'),min_threshold=('threshold','min'),max_threshold=('threshold','max'),old_raw_confidence_mean=('raw_confidence','mean'),decision_class_probability_mean=('decision_class_probability','mean'),decision_certainty_mean=('decision_certainty','mean')).reset_index())
audit.to_csv(OUTPUT_ROOT/'TABLE_confidence_alignment_audit.csv',index=False)
print('Overall mismatch:',f"{pred.argmax_vs_operational_mismatch.mean():.3%}")
display(audit.round(4))

In [ ]:
def safe_auroc(y,s):
    y=np.asarray(y,int)
    return np.nan if np.unique(y).size<2 else float(roc_auc_score(y,s))
def safe_auprc(y,s):
    y=np.asarray(y,int)
    return np.nan if np.unique(y).size<2 else float(average_precision_score(y,s))
def holm_adjust(ps):
    p=np.asarray(ps,float); out=np.full_like(p,np.nan); valid=np.where(np.isfinite(p))[0]
    if not len(valid): return out
    order=valid[np.argsort(p[valid])]; m=len(order); prev=0.0
    for rank,idx in enumerate(order):
        adj=max(prev,min(1.0,(m-rank)*p[idx])); out[idx]=adj; prev=adj
    return out
def new_rf(seed): return RandomForestClassifier(random_state=seed,**RF_PARAMS)
def make_failure_splits(g,strategy,n_splits=N_META_SPLITS):
    y=g.wrong.to_numpy(int)
    if strategy=='record':
        sp=StratifiedKFold(n_splits=n_splits,shuffle=True,random_state=RANDOM_STATE); return list(sp.split(np.zeros(len(g)),y))
    groups=g.pair_id.astype(str).to_numpy() if strategy=='pair' else (g.dataset.astype(str)+'|p'+g.p_idx.astype(str)).to_numpy()
    n_splits=min(n_splits,len(np.unique(groups)))
    sp=StratifiedGroupKFold(n_splits=n_splits,shuffle=True,random_state=RANDOM_STATE); return list(sp.split(np.zeros(len(g)),y,groups))
def failure_metrics(y,risk):
    y=np.asarray(y,int); risk=np.asarray(risk,float); order=np.argsort(-risk); k=max(1,int(math.ceil(.10*len(y)))); base=float(y.mean()); top=float(y[order[:k]].mean())
    return {'n':len(y),'baseline_failure_rate':base,'failure_AUROC':safe_auroc(y,risk),'failure_AUPRC':safe_auprc(y,risk),'top10_failure_rate':top,'top10_enrichment':float(top/max(base,EPS))}
def run_oof_rf(g,features,splits):
    X=g[features].to_numpy(float); y=g.wrong.to_numpy(int); oof=np.full(len(g),np.nan)
    for fid,(tr,te) in enumerate(splits):
        clf=new_rf(RANDOM_STATE+fid); clf.fit(X[tr],y[tr]); oof[te]=clf.predict_proba(X[te])[:,1]
    assert np.isfinite(oof).all(); return oof
def selective_metrics(g,risk):
    order=np.argsort(np.asarray(risk,float)); rows=[]
    for cov in COVERAGES:
        k=max(1,int(math.ceil(cov*len(g)))); a=g.iloc[order[:k]]; y=a.y.to_numpy(int); p=a.raw_probability.to_numpy(float); yh=a.raw_prediction.to_numpy(int)
        rows.append({'coverage':cov,'accepted_n':k,'AUROC':safe_auroc(y,p),'AUPRC':safe_auprc(y,p),'MCC':float(matthews_corrcoef(y,yh)),'ERROR':float(np.mean(y!=yh))})
    return rows

In [ ]:
HC_ROOT=OUTPUT_ROOT/'01_high_confidence'; HC_ROOT.mkdir(exist_ok=True)
rows=[]
for keys,g in pred.groupby(['dataset','model','seed','fold','run_id'],sort=True):
    dataset,model,seed,fold,run_id=keys
    old=g.raw_confidence.to_numpy(float)>=.90; new=g.decision_class_probability.to_numpy(float)>=.90
    rows.append({'dataset':dataset,'model':model,'seed':seed,'fold':fold,'run_id':run_id,'old_HC90_coverage':old.mean(),'old_HC90_error':float(g.loc[old,'wrong'].mean()) if old.any() else np.nan,'aligned_HC90_coverage':new.mean(),'aligned_HC90_error':float(g.loc[new,'wrong'].mean()) if new.any() else np.nan})
hc_run=pd.DataFrame(rows); hc_run.to_csv(HC_ROOT/'TABLE_HC90_run_level_old_vs_aligned.csv',index=False)
hc_summary=hc_run.groupby(['dataset','model'])[['old_HC90_coverage','old_HC90_error','aligned_HC90_coverage','aligned_HC90_error']].agg(['mean','std'])
hc_summary.to_csv(HC_ROOT/'TABLE_HC90_summary_old_vs_aligned.csv')
display(hc_summary.round(4))

In [ ]:
ABL_ROOT=OUTPUT_ROOT/'02_failure_ablation'; ABL_ROOT.mkdir(exist_ok=True)
ABLATION_FEATURES={'decision_certainty':['decision_certainty'],'decision_certainty+support':['decision_certainty','log1p_train_rna_degree'],'decision_certainty+ood':['decision_certainty','rna_centroid_cosine','protein_centroid_cosine','rna_diagonal_ood','protein_diagonal_ood'],'decision_certainty+support+ood':['decision_certainty','log1p_train_rna_degree','rna_centroid_cosine','protein_centroid_cosine','rna_diagonal_ood','protein_diagonal_ood']}
FINAL_FEATURE_SET='decision_certainty+support+ood'; FINAL_FEATURES=ABLATION_FEATURES[FINAL_FEATURE_SET]
result_rows=[]; score_rows=[]; sensitivity_rows=[]
for (dataset,model),g0 in pred.groupby(['dataset','model'],sort=True):
    g=g0.reset_index(drop=True).copy(); y=g.wrong.to_numpy(int); splits=make_failure_splits(g,'protein')
    for name,features in ABLATION_FEATURES.items():
        risk=run_oof_rf(g,features,splits); result_rows.append({'dataset':dataset,'model':model,'feature_set':name,'split_strategy':'protein_grouped',**failure_metrics(y,risk)})
        z=g[['dataset','model','seed','fold','run_id','row_id','pair_id','p_idx','y','wrong','raw_probability','raw_prediction','threshold','decision_class_probability','decision_certainty']].copy(); z['feature_set']=name; z['failure_risk']=risk; score_rows.append(z)
    sens=['decision_class_probability','log1p_train_rna_degree','rna_centroid_cosine','protein_centroid_cosine','rna_diagonal_ood','protein_diagonal_ood']
    sr=run_oof_rf(g,sens,splits); sensitivity_rows.append({'dataset':dataset,'model':model,'certainty_definition':'decision_class_probability',**failure_metrics(y,sr)})
    print('Finished',dataset,model)
failure_ablation=pd.DataFrame(result_rows); failure_ablation_oof=pd.concat(score_rows,ignore_index=True); certainty_sensitivity=pd.DataFrame(sensitivity_rows)
base=failure_ablation[failure_ablation.feature_set=='decision_certainty'][['dataset','model','failure_AUROC','failure_AUPRC']].rename(columns={'failure_AUROC':'baseline_AUROC','failure_AUPRC':'baseline_AUPRC'})
failure_ablation=failure_ablation.merge(base,on=['dataset','model']); failure_ablation['delta_AUROC_vs_certainty']=failure_ablation.failure_AUROC-failure_ablation.baseline_AUROC; failure_ablation['delta_AUPRC_vs_certainty']=failure_ablation.failure_AUPRC-failure_ablation.baseline_AUPRC
main=failure_ablation[failure_ablation.feature_set==FINAL_FEATURE_SET][['dataset','model','failure_AUROC','failure_AUPRC','top10_failure_rate','top10_enrichment']].copy(); main['certainty_definition']='normalized_threshold_margin'; certainty_sensitivity=pd.concat([main,certainty_sensitivity],ignore_index=True)
failure_ablation.to_csv(ABL_ROOT/'TABLE_failure_ablation_threshold_aligned.csv',index=False); failure_ablation_oof.to_csv(ABL_ROOT/'failure_ablation_oof_threshold_aligned.csv.gz',index=False,compression='gzip'); certainty_sensitivity.to_csv(ABL_ROOT/'TABLE_certainty_definition_sensitivity.csv',index=False)
display(failure_ablation.round(4)); print('Sensitivity'); display(certainty_sensitivity.round(4))

In [ ]:
LEAK_ROOT=OUTPUT_ROOT/'03_secondary_leakage'; LEAK_ROOT.mkdir(exist_ok=True)
rows=[]
for (dataset,model),g0 in pred.groupby(['dataset','model'],sort=True):
    g=g0.reset_index(drop=True).copy(); y=g.wrong.to_numpy(int)
    for strategy in ['record','pair','protein']:
        risk=run_oof_rf(g,FINAL_FEATURES,make_failure_splits(g,strategy)); rows.append({'dataset':dataset,'model':model,'feature_set':FINAL_FEATURE_SET,'split_strategy':strategy,**failure_metrics(y,risk)})
leakage_results=pd.DataFrame(rows)
ref=leakage_results[leakage_results.split_strategy=='protein'][['dataset','model','failure_AUROC']].rename(columns={'failure_AUROC':'protein_grouped_AUROC'})
leakage_results=leakage_results.merge(ref,on=['dataset','model']); leakage_results['AUROC_minus_protein_grouped']=leakage_results.failure_AUROC-leakage_results.protein_grouped_AUROC
leakage_results.to_csv(LEAK_ROOT/'TABLE_secondary_split_threshold_aligned.csv',index=False); display(leakage_results.round(4))

In [ ]:
LODO_ROOT=OUTPUT_ROOT/'04_lodo'; LODO_ROOT.mkdir(exist_ok=True)
rows=[]
for model,gm in pred.groupby('model',sort=True):
    datasets=sorted(gm.dataset.unique())
    for target in datasets:
        tr=gm[gm.dataset!=target]; te=gm[gm.dataset==target]; clf=new_rf(RANDOM_STATE); clf.fit(tr[FINAL_FEATURES].to_numpy(float),tr.wrong.to_numpy(int)); risk=clf.predict_proba(te[FINAL_FEATURES].to_numpy(float))[:,1]
        rows.append({'model':model,'target_dataset':target,'train_datasets':' + '.join(d for d in datasets if d!=target),'feature_set':FINAL_FEATURE_SET,**failure_metrics(te.wrong.to_numpy(int),risk)})
lodo_results=pd.DataFrame(rows); lodo_results.to_csv(LODO_ROOT/'TABLE_lodo_threshold_aligned.csv',index=False); display(lodo_results.round(4))

In [ ]:
RISK_ROOT=OUTPUT_ROOT/'05_risk_abstention'; RISK_ROOT.mkdir(exist_ok=True)
learned=failure_ablation_oof[failure_ablation_oof.feature_set==FINAL_FEATURE_SET].copy(); assert len(learned)==len(pred)
rows=[]
for run_id,g0 in learned.groupby('run_id',sort=True):
    g=g0.reset_index(drop=True); common={'dataset':g.dataset.iloc[0],'model':g.model.iloc[0],'seed':int(g.seed.iloc[0]),'fold':int(g.fold.iloc[0]),'run_id':run_id}
    for r in selective_metrics(g,1-g.decision_certainty.to_numpy(float)): rows.append({**common,'risk_method':'threshold_aligned_certainty',**r})
    for r in selective_metrics(g,g.failure_risk.to_numpy(float)): rows.append({**common,'risk_method':'learned_failure_risk',**r})
risk_coverage=pd.DataFrame(rows); risk_coverage.to_csv(RISK_ROOT/'risk_coverage_threshold_aligned.csv',index=False)
compare=[]
for (dataset,model,cov),dg in risk_coverage.groupby(['dataset','model','coverage']):
    a=dg[dg.risk_method=='threshold_aligned_certainty'][['seed','fold','MCC','ERROR']].rename(columns={'MCC':'MCC_base','ERROR':'ERROR_base'}); b=dg[dg.risk_method=='learned_failure_risk'][['seed','fold','MCC','ERROR']].rename(columns={'MCC':'MCC_learned','ERROR':'ERROR_learned'}); m=a.merge(b,on=['seed','fold'])
    for metric in ['MCC','ERROR']:
        x=m[f'{metric}_learned'].to_numpy(float); y=m[f'{metric}_base'].to_numpy(float); valid=np.isfinite(x)&np.isfinite(y); stat,pv=(wilcoxon(x[valid],y[valid],zero_method='wilcox') if valid.sum()>=3 and np.any(np.abs(x[valid]-y[valid])>0) else (np.nan,np.nan)); compare.append({'dataset':dataset,'model':model,'coverage':cov,'metric':metric,'n_runs':valid.sum(),'certainty_mean':np.nanmean(y),'learned_risk_mean':np.nanmean(x),'delta_learned_minus_certainty':np.nanmean(x-y),'wilcoxon_stat':stat,'p_value':pv})
risk_compare=pd.DataFrame(compare); risk_compare['p_holm']=np.nan
for _,idx in risk_compare.groupby(['dataset','coverage','metric']).groups.items(): risk_compare.loc[idx,'p_holm']=holm_adjust(risk_compare.loc[idx,'p_value'].to_numpy(float))
risk_compare.to_csv(RISK_ROOT/'TABLE_learned_vs_threshold_certainty.csv',index=False); display(risk_compare[np.isclose(risk_compare.coverage,.5)].sort_values(['dataset','metric','model']).round(4))

In [ ]:
# Compact key summary
summary=failure_ablation[failure_ablation.feature_set.isin(['decision_certainty',FINAL_FEATURE_SET])].pivot(index=['dataset','model'],columns='feature_set',values='failure_AUROC').reset_index().rename(columns={'decision_certainty':'failure_AUROC_certainty_only',FINAL_FEATURE_SET:'failure_AUROC_final'})
summary['failure_AUROC_gain']=summary.failure_AUROC_final-summary.failure_AUROC_certainty_only
leak=leakage_results.pivot(index=['dataset','model'],columns='split_strategy',values='failure_AUROC').reset_index().rename(columns={'record':'AUROC_record','pair':'AUROC_pair','protein':'AUROC_protein'}); leak['pair_minus_protein']=leak.AUROC_pair-leak.AUROC_protein
summary=summary.merge(leak,on=['dataset','model'])
ld=lodo_results[['model','target_dataset','failure_AUROC']].rename(columns={'target_dataset':'dataset','failure_AUROC':'LODO_failure_AUROC'}); summary=summary.merge(ld,on=['dataset','model'])
rr=risk_compare[np.isclose(risk_compare.coverage,.5)].pivot(index=['dataset','model'],columns='metric',values='delta_learned_minus_certainty').reset_index().rename(columns={'MCC':'learned_minus_certainty_MCC_at50','ERROR':'learned_minus_certainty_ERROR_at50'}); summary=summary.merge(rr,on=['dataset','model'])
summary.to_csv(OUTPUT_ROOT/'TABLE_threshold_aligned_key_summary.csv',index=False); display(summary.round(4))

In [ ]:
# Reproducibility config + compact ZIP
import sklearn, scipy, matplotlib
config={'python':sys.version,'platform':platform.platform(),'numpy':np.__version__,'pandas':pd.__version__,'scikit_learn':sklearn.__version__,'scipy':scipy.__version__,'matplotlib':matplotlib.__version__,'artifact_source':str(ARTIFACT_SOURCE),'prediction_rows':len(pred),'primary_run_ids':pred.run_id.nunique(),'argmax_vs_operational_mismatch_rate':float(pred.argmax_vs_operational_mismatch.mean()),'decision_class_probability':'p if operational prediction=1 else 1-p','decision_certainty':'normalized distance from validation-selected threshold on predicted side','rf_params':RF_PARAMS,'final_failure_features':FINAL_FEATURES}
with open(OUTPUT_ROOT/'analysis_config.json','w') as f: json.dump(config,f,indent=2)
manifest=[]
for q in sorted(OUTPUT_ROOT.rglob('*')):
    if q.is_file(): manifest.append({'relative_path':str(q.relative_to(OUTPUT_ROOT)),'size_bytes':q.stat().st_size})
pd.DataFrame(manifest).to_csv(OUTPUT_ROOT/'MANIFEST.csv',index=False)
zip_file=shutil.make_archive('/kaggle/working/RPI_Reliability_Threshold_Aligned_ARTIFACTS','zip',root_dir=OUTPUT_ROOT)
print('Created:',zip_file,'MB=',round(Path(zip_file).stat().st_size/1024**2,2))